# [실습] 모델 해석 · SHAP

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🔍 [실습] 모델 해석과 SHAP — 예측의 이유를 설명한다

## — 조립된 `Pipeline`을 열어, 예측의 이유를 사람의 말로 옮긴다

지금까지 다섯 순서에 걸쳐 모델을 만들고 검증했습니다. 성능은 AP 0.7543까지 왔고, 운영 임계값도 정했습니다. 그런데 아직 한 번도 답하지 못한 질문이 있습니다 — **"왜 이 세션을 구매로 봤나?"**

오늘 그 답을 만듭니다. 그리고 우리 모델은 전처리가 들어간 **`Pipeline`** 이라, SHAP을 걸기 전에 넘어야 할 문턱이 하나 있습니다.

> 🤖 **오늘의 AI 활용 규칙 — 협업 단계:**  
> SHAP 코드는 AI와 함께 써도 됩니다. 단, **해석 문장의 근거와 한계는 학습자가 직접 검증합니다.** 그리고 AI가 만든 해석에는 "설명 ≠ 인과"가 빠지는 경우가 많으니, 그 한계를 직접 확인해 기록합니다. (자세한 규칙은 개념 노트북 Part 0)

## 📋 오늘의 미션

경영진 보고가 잡혔습니다.

> 🧑‍💼 "모델이 잘 맞힌다는 건 알겠습니다. 그런데 임원 보고에서 **'모델이 어떤 관측값에 의존해 구매 확률을 높게 계산했나'** 를 설명해야 합니다. 그리고 특정 세션을 왜 그렇게 판단했는지도 예시로 두 건 준비해 주세요. **숫자의 단위와 한계를 포함한 문장으로** 부탁합니다."

| 문제 | 내용                                     | 확인하는 힘                           |
| ---- | ---------------------------------------- | ------------------------------------- |
| 1    | `Pipeline`에서 피처명을 복원해 SHAP 적용 | 세 단계 분해                          |
| 2    | Global 해석 Top 5 — 크기와 관찰 패턴     | 플롯과 수치 함께 읽기                 |
| 3    | Local 설명 2건 + 합산성 검산             | 한 건을 설명하기                      |
| 4    | 점검 — SHAP 0 · 누수 의심 · 한계         | 해석의 한계 인식                      |
| 5    | 설명 리포트 · 모델 카드 v7               | 근거·한계·후속 가설 기록 (**제출물**) |

> ⚠️ **오늘의 문턱:**  
> 우리 모델은 `ColumnTransformer`를 통과하면서 열 이름을 잃습니다. 복원하지 않으면 SHAP 결과가 `feature_8`·`feature_18` 같은 번호로만 나오고, **어느 피처가 중요한지 자체를 잘못 읽게 됩니다.** 문제 1이 그 복원 절차입니다.
>
> 개념 노트북에서 `Pipeline` × SHAP은 **Part 6 `[선택 학습]`** 입니다. 개념을 이해하는 데 반드시 필요한 단계가 아니기 때문입니다. 그러나 이 Practice의 모델은 전처리가 붙은 `Pipeline`이므로 **실무에서는 피해 갈 수 없는 단계**입니다. Part 6을 건너뛰었다면 지금 먼저 읽고 오면 좋습니다.

# ⚙️ 데이터와 모델 준비

Practice에서는 D04~D06에서 사용한 **UCI Online Shoppers Purchasing Intention** 데이터를 이어서 사용합니다. 한 행은 온라인 쇼핑몰 방문 세션 1건입니다.

| 항목              | 내용                                                           |
| ----------------- | -------------------------------------------------------------- |
| 관측 단위         | 온라인 쇼핑몰 방문 세션 1건                                    |
| 데이터 크기       | 12,330행 × 입력 피처 17개 + 타깃 1개                           |
| 타깃              | `Revenue` — 구매 완료 `True`(1), 미구매 `False`(0)             |
| 클래스 분포       | 구매 1,908건(15.5%) · 미구매 10,422건(84.5%)                   |
| 피처 구성         | 수치형 10개 + `has_page_value` + 범주형 7개                    |
| 모델              | `RandomForestClassifier(n_estimators=300, min_samples_leaf=5)` |
| 이전 내부 CV 결과 | AP 0.7543 · F1 0.6548                                          |
| 출처·이용 조건    | UCI Dataset 468 · DOI `10.24432/C5F88Q` · CC BY 4.0            |

### 오늘의 설명 계약

개념 노트북과 **같은 설명 계약**을 씁니다 — SHAP 환경·출력 공간·배경 방식이 같고, 데이터와 모델만 바뀝니다.

| 항목        | 설정                                                                             |
| ----------- | -------------------------------------------------------------------------------- |
| SHAP 환경   | SHAP 0.50.0 · `TreeExplainer`                                                    |
| 설명 대상   | 클래스 1(`Revenue=1`)의 확률                                                     |
| 출력 공간   | 확률 — SHAP 값의 단위는 **확률 변화량(%p)**                                      |
| 배경 데이터 | 학습셋 변환 결과에서 뽑은 **100건** · `interventional`                           |
| 분석 표본   | 테스트셋 앞 500건 — 계산 편의 표본이며 대표성은 보장하지 않음                    |
| 피처명      | `get_feature_names_out()`으로 변환 후 이름·순서 복원 (개념 Part 6 `[선택 학습]`) |

즉 `SHAP=+0.08`은 **구매 확률을 기준값보다 8%p 높이는 방향으로 귀속된 기여**입니다.

> ⚠️ **주의하기**  
> 명시적 배경 데이터를 쓰는 `interventional` 계산은 트리 300그루 × 표본 500건에서 **1분 가까이 걸릴 수 있습니다.** 진행 표시줄이 끝까지 가기를 기다립니다. 그리고 변환 배열의 자료형을 단순화하기 위해 `OneHotEncoder(sparse_output=False)`와 `.astype(float)`을 씁니다 — 이 구현의 호환성 선택이며, SHAP이 항상 밀집 배열만 받는다는 뜻은 아닙니다.

> ⚠️ **예측 시점 감사**  
> UCI 설명에서 `PageValues`는 거래 완료와 관련해 계산됩니다. 여기서 파생한 `has_page_value`도 같은 위험을 공유합니다. 실제 실시간 예측에 사용하려면 예측 요청 시점에 두 값이 확정되는지 확인해야 합니다. 확인 전에는 이 결과를 배포 성능이나 인과적 액션 근거로 사용하지 않습니다.

▶️ **코드 실행하기 · 코드 셀 1 [C1]**

In [1]:
# ─────────────────────────────────────────────
# [C1] ⚙️ 데이터와 모델 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import shap
except ImportError:                          # 코랩 등에서 없으면 설치합니다
    import subprocess
    subprocess.run("pip install -q shap", shell=True)
    import shap

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]
CAT_COLS = ["Month", "OperatingSystems", "Browser", "Region",
            "TrafficType", "VisitorType", "Weekend"]

feat = shoppers.copy()
feat["has_page_value"] = (feat["PageValues"] > 0).astype(int)   # 순서 5에서 채택한 파생변수
FINAL_NUM = NUM_COLS + ["has_page_value"]
y = feat["Revenue"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    feat[FINAL_NUM + CAT_COLS], y, test_size=0.2,
    random_state=RANDOM_STATE, stratify=y)

pipe = Pipeline([
    ("pre", ColumnTransformer([
        ("num", "passthrough", FINAL_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10,
                              sparse_output=False), CAT_COLS),   # SHAP은 밀집 배열이 필요합니다
    ])),
    ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                   random_state=RANDOM_STATE, n_jobs=-1)),
]).fit(X_train, y_train)

print(f"테스트 정확도 = {pipe.score(X_test, y_test):.4f}")
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

내려받는 중… online_shoppers.zip
테스트 정확도 = 0.8978

→ 준비 완료. 이제 여러분 차례입니다.


# 문제 1. `Pipeline`에서 피처명을 복원해 SHAP을 적용한다

시작 전에 위 계약의 세 가지를 확인합니다 — **누구의 예측**(앞 500건) · **어떤 출력**(클래스 1 확률) · **무엇과 비교**(배경 100건).

적용 순서는 다음과 같습니다.

1. **분리** — `named_steps`로 전처리와 모델을 꺼냅니다.
2. **변환** — 학습된 전처리로 테스트 데이터를 `transform`합니다. `fit_transform`은 학습 당시 표현과 달라지며, 결과를 모델 선택·평가에 사용하면 평가 누수 위험도 생깁니다.
3. **복원** — `get_feature_names_out()`으로 이름과 순서를 복원합니다.

```
[문제 1]
1) `pipe`에서 전처리(`pre`)와 모델(`clf`)을 각각 꺼냅니다.
2) 테스트 데이터의 앞 500건을 `transform`합니다.
   → 계산 편의 표본이며 대표 표본을 보장하지 않습니다.
3) `get_feature_names_out()`으로 피처명을 복원하고 접두사를 정리합니다.
4) 배경 데이터 100건을 만들고 `TreeExplainer`를 개념 노트북과 같은 설정으로 만듭니다.
5) `explainer(변환배열)`을 호출해 `Explanation` 객체를 얻고, `.values`·`.base_values`의 모양과 `clf.classes_`를 출력합니다.
6) 클래스 1(`Revenue=1`) 축을 선택한 뒤 기준값 + SHAP 합이 `predict_proba`와 허용 오차 안에서 일치하는지 확인합니다.
```

> 🤔 **예상하기**  
> 입력 18개 열이 One-Hot 이후 몇 개 피처가 될지 예상합니다. 그리고 개념 노트북의 XGBoost에서는 `.values`가 **2차원**이었습니다. 랜덤포레스트에서도 2차원일까요?

▶️ **코드 실행하기 · 코드 셀 2 [C2]**

In [ ]:
# [C2] 문제 1. `Pipeline`에서 피처명을 복원해 SHAP을 적용한다
# ⌨️ 문제 1 — 분리 → transform → 이름 복원 → SHAP

# 여기에 코드를 작성하세요 (named_steps → transform → get_feature_names_out → 배경 → Explanation)

<details>
<summary>(클릭) 💡 힌트</summary>

- `pipe.named_steps["pre"]`·`pipe.named_steps["clf"]`로 단계를 꺼냅니다.
- `pre.transform(X)`입니다. `fit_transform`을 쓰면 테스트 데이터로 인코더를 다시 학습해 버립니다.
- 변환 결과를 `.astype(float)`으로 캐스팅합니다 — 그러지 않으면 `object` 타입이라 SHAP이 거부합니다.
- 이름은 `num__PageValues` 형태로 나옵니다. `n.split("__", 1)[1]`로 접두사만 떼면 됩니다.
- 배경 데이터도 **변환된 공간**에서 뽑아야 합니다 — `shap.sample(pre.transform(X_train).astype(float), 100, random_state=RANDOM_STATE)`.
- Explainer 설정은 개념 노트북 Part 3의 Explainer 생성 셀과 같습니다 — `data=background`, `feature_perturbation="interventional"`, `model_output="probability"`.
- `explainer(배열)`을 호출하면 `Explanation` 객체가 나옵니다. `.values`·`.base_values`·`.data`·`.feature_names`를 확인합니다.
- 분류기라 클래스 축이 하나 더 붙을 수 있습니다. 3차원이면 마지막 축이 클래스인지 `clf.classes_`와 대조하고 클래스 1의 위치를 고릅니다.

</details>

> 🎯 **[C2] 확인하기**  
> 18개 열이 **70개 피처**가 됐습니다. 그리고 세 가지를 확인해 둡니다.
>
> **① `.values`가 3차원 `(500, 70, 2)` 입니다.** 개념 노트북의 XGBoost에서는 2차원이었는데, 여기서는 **클래스 축이 하나 더** 붙었습니다. 같은 `Explanation` API를 써도 모델에 따라 축 개수가 달라집니다. `clf.classes_`가 `[0, 1]`이므로 마지막 축의 위치 1이 구매 클래스입니다. **모양을 먼저 찍어 보지 않고 인덱싱하면 여기서 조용히 틀립니다.**
>
> **② 기준값 0.1794는 학습셋 구매율 0.1547과 다릅니다.** 기준값은 "학습 데이터의 정답 비율"이 아니라 **배경 데이터 100건에 대한 모델 출력의 평균**입니다. 배경 집단을 바꾸면 이 값도 바뀌므로, 리포트에는 배경 데이터의 출처와 크기를 반드시 적습니다.
>
> **③ 합산성이 확률 공간에서 성립합니다.** 개념 노트북과 **같은 확률 단위**이므로 기여값을 그대로 "%p"로 읽어도 됩니다.
>
> 마지막으로 `fit_transform`을 썼다면 테스트 데이터로 전처리기를 다시 적합해 학습 당시 표현과의 정합성이 깨집니다. 그 결과를 모델 선택·평가에 쓰면 평가 누수 위험도 생깁니다.

# 문제 2. Global 해석 Top 5 — 크기와 **방향**을 함께 읽는다

경영진의 첫 질문을 모델 설명이 답할 수 있는 범위로 바꿉니다 — **"모델은 어떤 관측값에 의존해 구매 확률을 높게 계산했는가?"**

개념 노트북 Part 2에서 세 방법을 비교했습니다 — **Gain(학습에 얼마나 쓰였나) · 순열 중요도(망가뜨리면 성능이 얼마나 떨어지나) · SHAP(출력을 어떻게 나누나)**. 앞의 둘은 부호가 없어 방향을 주지 못합니다. Local SHAP 값을 모아 만든 Global 해석은 **크기와 방향을 함께** 줍니다.

```
[문제 2]
1) 평균 |SHAP|으로 Global 해석 Top 8을 출력합니다.
2) summary plot(beeswarm)을 그립니다. 피처 이름이 제대로 나오는지 확인합니다.
3) Top 5 각각에 대해 피처값과 SHAP 값의 상관계수를 계산합니다.
   → 이 값은 선형 관계를 한 수치로 요약한 **보조 지표**입니다. 부호만으로 단조 방향을 확정하지 않고 Beeswarm과 구간별 패턴을 함께 확인합니다.
```

> 🤔 **예상하기**  
> 순서 5에서 만든 파생변수 `has_page_value`와, 순서 5에서 개방한 범주형 열들. **이들이 Top 5에 들어올까요?** 앞서 추가한 피처에 모델이 어떻게 의존하는지 여기서 확인합니다.

▶️ **코드 실행하기 · 코드 셀 3 [C3]**

In [ ]:
# [C3] 문제 2. Global 해석 Top 5 — 크기와 **방향**을 함께 읽는다
# ⌨️ 문제 2 — Global 해석(평균 |SHAP|) + summary plot + 방향 정량화

# 여기에 코드를 작성하세요 (평균 |SHAP| Top 8 → summary plot → 상관으로 방향)

<details>
<summary>(클릭) 💡 힌트</summary>

- Global 해석은 `pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names)`입니다.
- `shap.summary_plot(shap_values, X_sample_t, feature_names=feature_names, show=False)` — `feature_names`를 넘겨야 복원한 이름이 그림에 나옵니다.
- 선형 연관의 보조 요약은 열 하나를 골라 `np.corrcoef(X_sample_t[:, j], shap_values[:, j])[0, 1]`로 구합니다. `j`는 `feature_names.index(피처명)`.
- 분산이 0인 피처는 상관을 계산할 수 없으므로 별도로 표시합니다.

</details>

> 🎯 **[C3] 확인하기**  
> **앞선 순서에서 만든 두 피처가 이 모델의 예측에 어떻게 사용됐는지 확인했습니다.**
>
> **`has_page_value`가 2위(0.0829)** 입니다. 순서 5에서 "`PageValues`는 크기보다 0이냐 아니냐가 중요하다"는 가설로 만든 그 파생변수입니다. 단위가 확률이므로 **평균 약 8%p를 움직인 기여**로 읽을 수 있습니다. 다만 이것이 과거 F1 변화의 원인이라고 SHAP만으로 확정할 수는 없습니다.
>
> **`Month_Nov`가 3위(0.0194)** 입니다. 순서 5에서 추가한 범주형 표현 가운데 하나가 상위에 나타납니다. 해당 범주형 피처를 제외했다면 이 모델은 11월 여부를 직접 사용할 수 없었을 것입니다.
>
> 피처값-SHAP 상관과 Beeswarm을 함께 보면 `PageValues`·`has_page_value`·`Month_Nov`는 구매 확률 증가 기여와 양의 연관이, `ExitRates`·`BounceRates`는 음의 연관이 관찰됩니다. 이는 이 분석 표본의 모델 귀속 패턴이며 단조 효과나 인과 효과를 뜻하지 않습니다.

# 문제 3. Local 설명 두 건 — 합산성으로 검산하고, 문장으로 옮긴다

경영진의 두 번째 요청입니다. **"특정 세션을 왜 그렇게 판단했는지 예시 두 건."**

```
[문제 3]
1) 표본에서 예측 확률이 가장 높은 세션과 가장 낮은 세션을 각각 찾으세요.
2) 각 세션의 SHAP 기여 Top 5를 '기여도 + 변환된 피처값'과 함께 표로 출력합니다.
3) 합산성을 검산합니다 — `np.isclose`로 기준값 + 그 세션의 SHAP 합이 예측 확률과 허용 오차 안에서 같은지 확인합니다.
4) 두 세션에 대해 각각 '모델 근거 3가지 + 후속 검증 가설'을 문장으로 작성합니다.
```

> 🤔 **예상하기**  
> 확률이 가장 높은 세션과 가장 낮은 세션은 극단 사례이며 대표 사례가 아닙니다. **두 세션의 근거 피처가 같은 목록일까요?** 그리고 기여도의 *크기*는 어느 쪽이 클까요?

▶️ **코드 실행하기 · 코드 셀 4 [C4]**

In [ ]:
# [C4] 문제 3. Local 설명 두 건 — 합산성으로 검산하고, 문장으로 옮긴다
# ⌨️ 문제 3 — 고확률·저확률 세션의 Local 설명 + 합산성 검산

# 여기에 코드를 작성하세요 (최고·최저 확률 세션 → 기여 Top 5 표 → 합산성 검산)

<details>
<summary>(클릭) 💡 힌트</summary>

- 확률은 `pipe.predict_proba(X_sample)[:, 1]`입니다. 위치는 `np.argmax`·`np.argmin`으로 찾습니다.
- 한 세션의 기여는 `pd.Series(shap_values[i], index=feature_names)`입니다.
- 기여가 큰 순서로 뽑으려면 **절댓값**으로 정렬해야 합니다 — `c.reindex(c.abs().sort_values(ascending=False).index)`.
- 변환된 피처값은 `X_sample_t[i, feature_names.index(피처명)]`로 꺼냅니다. `shap_explanation.data`에서 꺼내도 같습니다.
- 합산성 검산: `np.isclose(base_value + shap_values[i].sum(), proba[i], atol=1e-6)`로 부동소수점 허용 오차 안에서 확인합니다.

</details>

> 🎯 **[C4] 확인하기**  
> 두 세션의 근거 피처 **목록은 거의 같고, 부호가 반대**입니다. 같은 피처가 어떤 세션에서는 확률을 올리고 다른 세션에서는 내립니다 — 이것이 Global 해석 하나로는 개별 세션을 설명할 수 없는 이유입니다.
>
> **기여의 크기는 고확률 세션이 압도적으로 큽니다.** 고확률 세션의 SHAP 합은 `+0.7445`인데 저확률 세션은 `−0.1794`입니다. 기준값이 0.1794이므로 확률을 0까지 내리는 데 쓸 수 있는 여지가 애초에 `−0.1794`뿐입니다. **확률 공간에서는 기여의 크기가 위아래로 비대칭**입니다.
>
> 합산성은 두 세션 모두 `predict_proba`와 정확히 일치합니다. 이것이 개념 노트북에서 손으로 검산한 그 성질이며, **설명이 모델을 실제로 재구성하고 있다는 증거**입니다.

# 문제 4. 점검 — SHAP 0, 누수 의심, 그리고 한계

해석 리포트에서 가장 자주 빠지는 것이 **점검과 한계**입니다. 세 가지를 확인합니다.

```
[문제 4]
1) SHAP이 정확히 0인 피처를 찾아 개수와 이름을 출력합니다.
   → 그 피처들의 공통점이 무엇인지 생각해 봅니다. "쓸모없다"고 결론 내려도 될까요?
2) 상위 피처 중 예상 밖으로 강한 피처가 있는지 확인하고 생성 시점·출처를 감사합니다.
3) 이름을 복원하지 않았다면 어떻게 보였을지 확인합니다.
   → Top 3의 열 번호와 복원된 이름을 나란히 출력합니다.
```

> 🤔 **예상하기**  
> 70개 피처 중 SHAP이 **정확히 0**인 것이 몇 개나 있을까요? 그리고 그런 피처는 어떤 종류일까요?
>
> 개념 노트북 Part 5의 세 가지 한계 가운데 이 데이터에서 가장 위험한 것도 함께 생각해 봅니다.

▶️ **코드 실행하기 · 코드 셀 5 [C5]**

In [ ]:
# [C5] 문제 4. 점검 — SHAP 0, 누수 의심, 그리고 한계
# ⌨️ 문제 4 — SHAP 0 피처 · 누수 점검 · 이름 복원 전/후 대조

# 여기에 코드를 작성하세요 (SHAP 0 목록 → 상위 피처 도메인 점검 → 번호 vs 이름 대조)

<details>
<summary>(클릭) 💡 힌트</summary>

- SHAP이 0인 피처는 `np.abs(shap_values).mean(axis=0) == 0`인 열입니다.
- 이름 복원 전 모습을 보려면 열 **번호**를 그대로 쓰면 됩니다 — `f"feature_{j}"`.
- Top 3의 열 번호는 `np.argsort(중요도)[::-1][:3]`으로 얻습니다.
- 누수 점검은 코드가 아니라 **판단**입니다. 상위 피처를 하나씩 보며 "이 값이 예측 시점에 확정되는가", "타깃 또는 이후 정보를 사용했는가"를 확인합니다.

</details>

> 🎯 **[C5] 확인하기**  
> **7개**이고, 전부 **희귀 범주**였습니다(특정 운영체제·특정 유입 경로). 표본 500건과 현재 설명 설정에서 기여가 0으로 귀속됐습니다. 희귀 범주와 표본 범위가 가능한 원인이지만 추가 확인이 필요합니다.
>
> 여기서 중요한 것은 결론을 어떻게 내리느냐입니다. **SHAP 0은 쓸모없다는 뜻이 아니라 이 모델·표본·설명 설정에서 귀속된 기여가 0** 입니다. 이 구분을 놓치면 피처를 성급하게 제거할 수 있습니다.
>
> 그리고 이름 복원의 값이 마지막에 드러납니다. 복원 안 했다면 보고서에 **"feature_8이 가장 중요합니다"** 라고 썼을 것입니다. 그 문장으로는 아무도 무엇을 해야 할지 모릅니다. **`get_feature_names_out()` 한 줄이 숫자를 보고서로 바꿉니다.**

# 문제 5. 설명 리포트 · 모델 카드 v7 — 오늘의 제출물

Global 해석과 Local 설명을 하나의 리포트로 묶습니다. 경영진이 읽는 문서이므로 **숫자보다 문장이 앞에** 와야 합니다.

```
[문제 5]
1) Global 해석 표를 CSV로 저장합니다 — shap_global_v7.csv
   (열: 순위, 피처, 평균 |SHAP|, 상관 부호(보조), 업무 해석)
2) 아래 설명 리포트와 모델 카드 v7 템플릿을 채우세요.
```

▶️ **코드 실행하기 · 코드 셀 6 [C6]**

In [ ]:
# [C6] 문제 5. 액션 번역 리포트 · 모델 카드 v7 — 오늘의 제출물
# ⌨️ 문제 5 — Global 해석 표를 파일로 남기기

# 여기에 코드를 작성하세요 (Top 5 + 방향을 표로 만들어 CSV 저장)

**설명 리포트 + 모델 카드 v7 템플릿** — 아래 빈칸을 채워 이 셀에 완성합니다.

```markdown
## Global 해석 (summary plot 기준)

| 순위 | 피처 | 평균 ` | SHAP                                        | `   | 관찰된 패턴 | 업무 해석 |
| ---- | ---- | ------ | ------------------------------------------- | --- | ----------- | --------- |
| 1    | { }  | { }    | 큰 값에서 구매 출력 증가 기여가 주로 관찰됨 | { } |
| 2    | { }  | { }    | { }                                         | { } |
| 3    | { }  | { }    | { }                                         | { } |
| 4    | { }  | { }    | 큰 값에서 구매 출력 감소 기여가 주로 관찰됨 | { } |
| 5    | { }  | { }    | { }                                         | { } |

## Local 설명 (2건)

### 고확률 세션 #{ } — 예측 확률 { }

- 근거 3가지: { 피처 }({ 기여 }) · { } · { }
- 후속 검증 가설: { 추가 분석 또는 실험 }

### 저확률 세션 #{ } — 예측 확률 { }

- 근거 3가지: { }
- 후속 검증 가설: { }

## 점검

- **SHAP 0인 피처: { 개수 }개 { 목록 } → 원인 추정: { }**
- **"쓸모없다"고 결론 내리지 않은 이유: { }**
- 시점 누수 감사 피처: `PageValues`, `has_page_value` → 예측 시점 가용성 확인 결과: { }
- **한계 명시: 설명 ≠ 인과 / 피처 구성이 바뀌면 순위도 바뀜 / 표본 { }건 기준**

## 모델 카드 v7 — 구매 전환 예측 (해석 추가)

- 모델·전처리: { 순서 5·6에서 채택한 구성 }
- 성능: CV AP { } · F1 { } · 운영 임계값 { }
- **설명 계약: SHAP {버전} · `TreeExplainer` · 클래스 1(`Revenue=1`) 확률 공간 · 배경 데이터 { }건(`interventional`) · 표본 { }건 · 기준값 { }**
- **피처명 복원: get_feature_names_out() 사용 여부 { } · 입력 { }열 → 변환 { }피처**
- **Global Top 5와 방향: { 위 표 요약 }**
- **Local 설명 2건 + 후속 검증 가설: { 위 요약 }**
- **해석의 한계: { 세 가지 }**
- AI 사용 내역: { 무엇을 시켰나 / 해석 문장은 누가 썼나 / "설명≠인과"를 직접 확인했나 }
```

**제출:** 노트북 · `shap_global_v7.csv` · 설명 리포트를 개인 공개 저장소 main에 커밋·푸시하고, 저장소·커밋 링크를 제출합니다.

**스스로 점검하는 기준**

| 축            | 기준                                                                          |
| ------------- | ----------------------------------------------------------------------------- |
| 방향 정확성   | 크기만이 아니라 어느 쪽으로 미는지를 읽었는가                                 |
| 단위·합산성   | SHAP을 확률 변화량(%p)으로 읽고 `기준값 + ΣSHAP = predict_proba`를 검산했는가 |
| 비즈니스 문장 | 기여도 숫자가 아니라 사람이 읽는 문장으로 옮겼는가                            |
| 한계 명시     | 설명 ≠ 인과, 피처 구성 의존성을 적었는가                                      |
| 피처명 복원   | `get_feature_names_out()`으로 이름을 되살렸는가                               |

> 🚀 **직접 확장하기**  
> `has_page_value`를 제외한 모델을 별도 검증 데이터에서 다시 평가하고 SHAP을 계산합니다. **`PageValues`의 기여가 얼마나 커지는지** 확인하면, 상관된 피처 구성이 귀속에 어떤 영향을 주는지 비교할 수 있습니다. 그리고 그것이 왜 **SHAP 순위를 절대 고정된 사실로 보고하면 안 되는지**의 이유입니다.

오늘 여러분은 조립된 `Pipeline`을 열어 예측의 이유를 꺼냈고, 그 이유를 경영진이 읽을 문장으로 옮겼습니다.

그리고 앞서 추가한 피처에 모델이 어떻게 의존했는지 확인했습니다 — 순서 5에서 만든 `has_page_value`가 Global 2위, 순서 5에서 개방한 범주형에서 나온 `Month_Nov`가 3위였습니다. **현재 분석 표본에서 두 피처에 비교적 큰 기여가 귀속됐습니다.**

설명의 출력 공간·표본·한계를 함께 기록하면 결과를 검토 가능한 형태로 남길 수 있습니다.

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>